# 04 — GNN Training & Evaluation

Train the GCN model for district scoring, evaluate against a weighted-sum baseline, and visualize predictions.

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
import torch

from kg.config import NUM_DISTRICTS, DISTRICTS_GEOJSON, FEATURES_OUTPUT
from kg.gnn import (
    build_dataset, generate_labels, DistrictGCN,
    train, predict, _load_features, NUMERIC_FEATURE_KEYS,
)

%matplotlib inline

## 1. Dataset & Model Training

In [ ]:
data = build_dataset()
features = _load_features()

print(f'Nodes: {data.num_nodes}')
print(f'Node feature dim: {data.x.size(1)}')
print(f'Edges: {data.edge_index.size(1)}')

model = train(data, features)

## 2. GNN vs. Weighted-Sum Baseline

The baseline computes district scores as a direct weighted sum of normalized features (no graph structure). We compare it against the GCN which incorporates neighborhood information via message passing.

In [ ]:
def baseline_scores(features: dict, pref: list[float]) -> list[tuple[int, float]]:
    """Weighted-sum baseline without graph structure."""
    raw = np.array([
        [features[str(d)][k] for k in ['avg_price_per_sqm', 'accessibility_score', 'transit_density']]
        for d in range(1, NUM_DISTRICTS + 1)
    ])
    mins, maxs = raw.min(axis=0), raw.max(axis=0)
    ranges = maxs - mins
    ranges[ranges == 0] = 1.0
    normed = (raw - mins) / ranges
    
    scores = pref[0] * (1 - normed[:, 0]) + pref[1] * normed[:, 1] + pref[2] * normed[:, 2]
    ranked = sorted([(d + 1, float(scores[d])) for d in range(NUM_DISTRICTS)], key=lambda x: x[1], reverse=True)
    return ranked


profiles = {
    'Budget commuter': [0.6, 0.3, 0.1],
    'Central living': [0.1, 0.7, 0.2],
    'Transit lover': [0.1, 0.2, 0.7],
    'Balanced': [0.33, 0.34, 0.33],
}

print(f'{"Profile":20s} | {"GNN Top 5":40s} | {"Baseline Top 5":40s}')
print('-' * 105)
for name, pref in profiles.items():
    gnn_ranked = predict(model, data, pref)
    bl_ranked = baseline_scores(features, pref)
    
    gnn_top = ', '.join(f'D{d}({s:.3f})' for d, s in gnn_ranked[:5])
    bl_top = ', '.join(f'D{d}({s:.3f})' for d, s in bl_ranked[:5])
    print(f'{name:20s} | {gnn_top:40s} | {bl_top:40s}')

## 3. Rank Correlation

In [ ]:
from scipy.stats import spearmanr

rng = np.random.RandomState(42)
correlations = []

for _ in range(50):
    pref = rng.dirichlet([1, 1, 1]).tolist()
    gnn_ranked = predict(model, data, pref)
    bl_ranked = baseline_scores(features, pref)
    
    gnn_order = [d for d, _ in gnn_ranked]
    bl_order = [d for d, _ in bl_ranked]
    
    rho, _ = spearmanr(
        [gnn_order.index(d) for d in range(1, NUM_DISTRICTS + 1)],
        [bl_order.index(d) for d in range(1, NUM_DISTRICTS + 1)],
    )
    correlations.append(rho)

print(f'Mean Spearman rho (GNN vs baseline): {np.mean(correlations):.3f} +/- {np.std(correlations):.3f}')

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(correlations, bins=20, color='steelblue', edgecolor='white')
ax.axvline(np.mean(correlations), color='red', linestyle='--', label=f'mean={np.mean(correlations):.3f}')
ax.set_xlabel('Spearman rho')
ax.set_ylabel('Count')
ax.set_title('GNN vs. Baseline Rank Correlation (50 random preferences)')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Score Maps

In [ ]:
districts_gdf = gpd.read_file(str(DISTRICTS_GEOJSON))
for col in ['BEZNR', 'BEZ', 'DISTRICT_CODE', 'district']:
    if col in districts_gdf.columns:
        districts_gdf['d_num'] = districts_gdf[col].astype(int)
        break

fig, axes = plt.subplots(2, 2, figsize=(14, 14))

for ax, (name, pref) in zip(axes.flat, profiles.items()):
    ranked = predict(model, data, pref)
    score_map = {d: s for d, s in ranked}
    districts_gdf['score'] = districts_gdf['d_num'].map(score_map)
    
    districts_gdf.plot(column='score', ax=ax, legend=True, cmap='RdYlGn',
                       edgecolor='white', linewidth=1.5)
    for _, row in districts_gdf.iterrows():
        centroid = row.geometry.centroid
        ax.annotate(str(row.d_num), (centroid.x, centroid.y),
                    ha='center', va='center', fontsize=9, fontweight='bold')
    ax.set_title(f'{name}\n(afford={pref[0]}, access={pref[1]}, transit={pref[2]})')
    ax.axis('off')

plt.tight_layout()
plt.show()

## 5. Qualitative Assessment

- The GNN generally agrees with the weighted-sum baseline on extreme preferences (e.g., pure affordability → outer districts, pure accessibility → central districts).
- Where they diverge, the GNN tends to boost districts with well-connected neighbors, reflecting the graph-based message passing.
- With only 23 nodes the GNN cannot learn dramatically different patterns from the baseline, but it captures inter-district relationships that the baseline ignores.
- The model is best viewed as a demonstration of GNN applicability on a knowledge graph rather than a production recommender.